In [47]:
import glob,codecs,os,yaml
import pandas as pd
from unidecode import unidecode
from os.path import expanduser
def ding():
    os.system('afplay /System/Library/Sounds/Submarine.aiff')
from IPython.core.display import HTML,display 

## ⚠️
Pour les cas où le genre des noms est marqué explicitement, on utilise *nomGenre=True*

In [48]:
%store -r numerosKalaba typeKalaba
# numerosKalaba=[1]
print numerosKalaba, typeKalaba
nomGenre=False

[9] Kanonik


In [49]:
debug=0

# -*- coding: utf8 -*-
home = expanduser("~")

if typeKalaba!="Kanonik":
    numeroKalaba="23-K%d/"%numerosKalaba[0]
    repertoire=home+"/ownCloud/Cours/Bordeaux/L1-LinguistiqueGenerale/00-ProjetKalaba/"
    serie=repertoire+numeroKalaba
else:
    repertoire=home+"/ownCloud/Cours/Bordeaux/L1-UE4-Morphologie/00-Kanonik/"
    serie=repertoire+"24-Kanoniks/"
    serie=serie+"Kanonik-%02d/"%numerosKalaba[0]

print serie

/Volumes/BroadExt/ownCloud-Bdx3/Cours/Bordeaux/L1-UE4-Morphologie/00-Kanonik/24-Kanoniks/Kanonik-09/


In [50]:
nomDeclarationRad="Declarations-Radicaux.tex"
nomDeclarationDec="Declarations-Decoupages.tex"
nomTableauxRad="Tableaux-Gloses.yaml"
nomDeclaration="Declarations.tex"
nomTableaux="Tableaux.yaml"

In [51]:
flexion=pd.read_csv(serie+"Clozes.txt",sep=";",header=None, names=list(range(20)),encoding="utf8").dropna(axis='columns', how='all')
flexion.head()

,0,1,2,3,5,6,7,8,9,10,11,12,13
0,#\tNOM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,#,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,#,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,caillouISg,caillou,NOM,tari,tarigi,tari-gi,"NOM, CF=N1, Genre=I, Nombre=Sg",caillou.N1.I-N1.Sg,3.0,caillou,.N1.I,-N1.Sg,NaN
4,caillouIDu,caillou,NOM,tari,tarima,tari-ma,"NOM, CF=N1, Genre=I, Nombre=Du",caillou.N1.I-N1.Du,3.0,caillou,.N1.I,-N1.Du,NaN


In [52]:
categories=flexion[2].dropna().unique().tolist()
categories

[u'NOM', u'VER', u'PRO', u'DET', u'ADJ', u'PREP']

bFlex sélectionne les lignes non-vides correspondant à une catégorie

In [53]:
bFlex={c:(flexion[7].notnull()) & (flexion[2]==c.upper()) for c in categories}
flexion[bFlex["NOM"]]

,0,1,2,3,5,6,7,8,9,10,11,12,13
3,caillouISg,caillou,NOM,tari,tarigi,tari-gi,"NOM, CF=N1, Genre=I, Nombre=Sg",caillou.N1.I-N1.Sg,3.0,caillou,.N1.I,-N1.Sg,NaN
4,caillouIDu,caillou,NOM,tari,tarima,tari-ma,"NOM, CF=N1, Genre=I, Nombre=Du",caillou.N1.I-N1.Du,3.0,caillou,.N1.I,-N1.Du,NaN
5,caillouIPl,caillou,NOM,tari,tarilo,tari-lo,"NOM, CF=N1, Genre=I, Nombre=Pl",caillou.N1.I-N1.Pl,3.0,caillou,.N1.I,-N1.Pl,NaN
6,greveISg,grève,NOM,vetub,vetubgi,vetub-gi,"NOM, CF=N1, Genre=I, Nombre=Sg",grève.N1.I-N1.Sg,3.0,grève,.N1.I,-N1.Sg,NaN
7,greveIDu,grève,NOM,vetub,vetubma,vetub-ma,"NOM, CF=N1, Genre=I, Nombre=Du",grève.N1.I-N1.Du,3.0,grève,.N1.I,-N1.Du,NaN
8,greveIPl,grève,NOM,vetub,vetublo,vetub-lo,"NOM, CF=N1, Genre=I, Nombre=Pl",grève.N1.I-N1.Pl,3.0,grève,.N1.I,-N1.Pl,NaN
9,milieuISg,milieu,NOM,jukar,jukargi,jukar-gi,"NOM, CF=N1, Genre=I, Nombre=Sg",milieu.N1.I-N1.Sg,3.0,milieu,.N1.I,-N1.Sg,NaN
10,milieuIDu,milieu,NOM,jukar,jukarma,jukar-ma,"NOM, CF=N1, Genre=I, Nombre=Du",milieu.N1.I-N1.Du,3.0,milieu,.N1.I,-N1.Du,NaN
11,milieuIPl,milieu,NOM,jukar,jukarlo,jukar-lo,"NOM, CF=N1, Genre=I, Nombre=Pl",milieu.N1.I-N1.Pl,3.0,milieu,.N1.I,-N1.Pl,NaN
12,fermeISg,ferme,NOM,rekam,rekamgi,rekam-gi,"NOM, CF=N1, Genre=I, Nombre=Sg",ferme.N1.I-N1.Sg,3.0,ferme,.N1.I,-N1.Sg,NaN


### Analyse de la case
on fait un dict à partir de la case pour éclater en colonne puis faire un pivot_table

In [54]:
def makeParadigme(row):
    result={}
    parts=row["Case"].split(", ")
    for part in parts[1:]:
        attr,val=part.split("=")
        result[attr.strip()]=val
    return result

# Tableaux pour une catégorie

In [55]:
def reindexNombre(tableau):
    result=tableau
    if "Du" in tableau.columns:
        result=tableau.reindex("Sg Du Pl".split(" "),axis=1)
    elif "Pau" in tableau.columns:
        result=tableau.reindex("Sg Pau Pl".split(" "),axis=1)
    elif "Sg" in tableau.columns:
        result=tableau.reindex("Sg Pl".split(" "),axis=1)
    return result

def reindexPers(tableau):
    result=tableau
    if "3Du" in tableau.columns:
        result=tableau.reindex("3Sg 3Du 3Pl".split(" "),axis=1)
    elif "3Pau" in tableau.columns:
        result=tableau.reindex("3Sg 3Pau 3Pl".split(" "),axis=1)
    elif "3Sg" in tableau.columns:
        result=tableau.reindex("3Sg 3Pl".split(" "),axis=1)
    return result

def reindexTableau(tableau):
    return reindexNombre(reindexPers(tableau))

In [56]:
def makeTableaux(cat):
    # sel sélectionne la catégorie cat dans flexion
    # les colonnes 1, 6 et 7 correspondent à la catégorie, la forme et la case
    sel=bFlex[cat]
    catTableaux={}
    catCF={}
    radCF={}
    cfLexemes={}

    # la colonnne 3 contient le radical
    lDF=flexion[sel][[1,6,7]]
    lDF=flexion[sel][[1,3,6,7]]
    lDF.columns=(u"%s Radical Forme Case"%cat).split(" ")
    # on ajoute une colonne avec un dict correspondant à attribut:valeur
    lDF["dCell"]=lDF.apply(makeParadigme,axis=1)
    if lDF.iloc[0]["dCell"]!={}:
        # on éclate le dict attribut:valeur en colonnes
        # on calcule les dimensions du tableau croisé pour pivot_table
        dfCat=pd.concat([lDF.drop(['Case',"dCell"], axis=1), lDF['dCell'].apply(pd.Series)], axis=1)
        setColonnes=set(dfCat.columns.tolist())
        if cat=="NOM" and not nomGenre:
            setColonnes-=set(["Genre"])
        if "Nombre" in setColonnes:
            colonnes="Nombre"
            lignes=setColonnes-set([cat,"Radical","Forme","CF","Nombre"])
        elif "Pers" in setColonnes:
            colonnes="Pers"
            lignes=list(setColonnes-set([cat,"Radical","Forme","CF","Pers"]))
        if debug: print colonnes, lignes
        # on fait la liste des lexèmes de la catégorie cat
        listCat=dfCat[cat].unique().tolist()
        if debug: display(dfCat.head())
        for l in listCat:
            print l
            catL=dfCat[dfCat[cat].str.encode("utf8")==l.encode("utf8")]
            radical=catL["Radical"].values[0]
            if "CF" in dfCat.columns:
                lCF=catL["CF"].values[0]
                if lCF not in cfLexemes:
                    cfLexemes[lCF]=[]
                cfLexemes[lCF].append(l)
                #
                # reindexNombre présente les colonnes concernant le nombre dans l'ordre traditionnel
                # SG < DU/PAU < PL
                #
                if lCF not in catCF:
                    catTableaux[lCF]=pd.pivot_table(catL,index=lignes,columns=colonnes,values="Forme", aggfunc='first')
                    catCF[lCF]=l
                    radCF[lCF]=radical
            elif cat=="NOM" and nomGenre:
                lCF=catL["Genre"].values[0]
                if lCF not in cfLexemes:
                    cfLexemes[lCF]=[]
                cfLexemes[lCF].append(l)
                if lCF not in catCF:
                    catTableaux[lCF]=pd.pivot_table(catL,index=lignes,columns=colonnes,values="Forme", aggfunc='first')
                    catCF[lCF]=l
                    radCF[lCF]=radical
            else:
                catCF[cat]=l
                radCF[cat]=radical
                catTableaux[cat]=pd.pivot_table(catL,index=lignes,columns=colonnes,values="Forme", aggfunc='first')
        for t in catTableaux:   
            if debug: display(HTML("<h1>%s, %s</h1>"%(t,catCF[t])),reindexTableau(catTableaux[t]))
#             print t,catCF[t],radCF[t]
#             print reindexTableau(catTableaux[t]).to_latex(multirow=True)
    return catTableaux,cfLexemes,catCF,radCF

In [57]:
tableaux={}
tableauxLatex=[]
lexemesCF={}
cfLexs={}
cfRads={}
tableauxLatex.append(ur"\subsection*{Tableaux de flexion}")
for cat in "DET PRO NOM ADJ VER".split(" "):
#    print ur"\section{%s}"%cat
    tableauxLatex.append(ur"\subsubsection*{%s}"%cat)
    tableaux[cat],lexemesCF[cat],cfLexs[cat],cfRads[cat]=makeTableaux(cat)
#     print tableaux[cat]
#     print lexemesCF[cat]
    for t in tableaux[cat]:
#        print t
        tableauxLatex.append(u"\\needspace{6\\baselineskip}")
        tableauxLatex.append(u"\\noindent\n%s %s => %s\\\\"%(t,cfLexs[cat][t],cfRads[cat][t]))
        tableauxLatex.append(reindexTableau(tableaux[cat][t]).to_latex(multirow=True)+u" \\\\ \\medskip")
        tableauxLatex.append("")
print "\n".join(tableauxLatex)

DEF
IND
DEM
PRO
caillou
grève
milieu
ferme
fromage
cause
autorisation
oasis
désert
Camille
éleveur
fermière
Aloys
habitant
immortelle
chèvre
poussin
grenouille
fantôme
animal
rivière
souffrance
lâcheté
katana
guerre
contrée
peur
mort
point
employé
humain
Rachel
survivante
Maurine
escargot
tortue
cactus
cochon
atroce
jaune
courageux
faible
violet
beau
vide
grand
ridicule
rouge
lointain
frais
adorer
puer
voir
appartenir
soigner
lancer
élever
trouver
faire
décéder
venir
commencer
tomber
devenir
détester
gambader
chasser
attirer
vivre
attraper
fuir
soutenir
vendre
mourir
manger
être
venger
immoler
\subsection*{Tableaux de flexion}
\subsubsection*{DET}
\needspace{6\baselineskip}
\noindent
DET DEM => d\\
\begin{tabular}{llll}
\toprule
Nombre &      Sg &      Du &      Pl \\
Genre &         &         &         \\
\midrule
A     &  d-u-to &  d-u-ti &  d-u-te \\
H     &  d-a-to &  d-a-ti &  d-a-te \\
I     &  d-i-to &  d-i-ti &  d-i-te \\
\bottomrule
\end{tabular}
 \\ \medskip

\subsubsection*{

# Liste des lexèmes par CF

=> à intégrer aux tableaux...

In [58]:
cfLatex=[]
cfLatex.append(ur"\subsection*{Classes flexionnelles}")
for cat in "NOM ADJ VER DET".split(" "):
    classeCat=lexemesCF[cat]
    if classeCat:
        if debug: print cat+" : "
        cfLatex.append(cat+" : ")
        if debug: print ur"\begin{description}"
        cfLatex.append(ur"\begin{description}")
        for c in classeCat:
            if debug: print ur"\item[- %s] %s"%(c,", ".join(sorted(classeCat[c])))
            cfLatex.append(ur"\item[- %s] %s"%(c,", ".join(sorted(classeCat[c]))))
        if debug: print ur"\end{description}"
        cfLatex.append(ur"\end{description}")
if len(cfLatex)<2:
    cfLatex=[]
print "\n".join(cfLatex)

\subsection*{Classes flexionnelles}
NOM : 
\begin{description}
\item[- N1] Aloys, Camille, animal, autorisation, caillou, cause, chèvre, désert, fantôme, ferme, fermière, fromage, grenouille, grève, habitant, immortelle, milieu, oasis, poussin, éleveur
\item[- N2] Maurine, Rachel, cactus, cochon, contrée, employé, escargot, guerre, humain, katana, lâcheté, mort, peur, point, rivière, souffrance, survivante, tortue
\end{description}


# Liste des noms par genre

- on ajoute une colonne genre en coupant dans la matrice de traits
- on fait la liste des genres
- on fait la liste des noms par genre
- on fabrique la structure LaTeX

In [59]:
genresLatex=[]

genresLatex.append(ur"\subsection*{Genre des noms}")

# Ajout de la colonne genre
flexion["genre"]=flexion[bFlex["NOM"]][7].str.extract("Genre=(.+?),")

# Liste des genres
genres=sorted([g for g in flexion["genre"].unique().tolist() if isinstance(g,unicode)])
genreNoms={}

# Liste des noms par genre
for g in genres:
    genreNoms[g]=flexion[flexion.genre==g][1].unique().tolist()

# LaTeX
if debug: print ur"\begin{description}"
genresLatex.append(ur"\begin{description}")
for g in genres:
    if debug: print ur"\item[- %s]"%g, ", ".join(sorted(genreNoms[g]))
    genresLatex.append(ur"\item[- %s] %s"%(g, ", ".join(sorted(genreNoms[g]))))
if debug: print ur"\end{description}"
genresLatex.append(ur"\end{description}")

print "\n".join(genresLatex)

\subsection*{Genre des noms}
\begin{description}
\item[- A] animal, cactus, chèvre, cochon, escargot, fantôme, grenouille, poussin, tortue
\item[- H] Aloys, Camille, Maurine, Rachel, employé, fermière, habitant, humain, immortelle, mort, survivante, éleveur
\item[- I] autorisation, caillou, cause, contrée, désert, ferme, fromage, grève, guerre, katana, lâcheté, milieu, mort, oasis, peur, point, rivière, souffrance
\end{description}


# Calcul des remplissages de paradigmes
- filledCells contient les informations exportées par Phrases2
- sortLevel permet de mettre les noms des traits dans l'ordre logique
- filledTable fait le tableau du remplissage pour une catégorie avec un pivot_table
    - on lui passe 
        - le filledCells de la catégorie
        - la colonne du lexème
        - les colonnes qu'on veut mettre en ligne dans le tableau du paradigme
        - les colonnes qu'on veut mettre en colonnes dans le tableau du paradigme

In [60]:
with open(serie+"FilledCells.yaml", 'r') as stream:
    filledCells=yaml.safe_load(stream)
filledCells["VER"]=[v.replace(".Trois",".3") for v in filledCells["VER"]]
filledCells

{'ADJ': ['courageux.H.SG',
  'atroce.I.PL',
  'violet.A.PL',
  'ridicule.A.SG',
  'atroce.A.PL',
  'beau.H.PL',
  'beau.A.DU',
  'grand.I.SG',
  'faible.I.SG',
  'beau.I.DU',
  'rouge.I.DU',
  'jaune.A.PL',
  'lointain.I.SG',
  'frais.I.DU',
  'vide.I.SG'],
 'DET': ['DEF.I.DU',
  'DEF.H.PL',
  'DEM.H.PL',
  'DEM.I.SG',
  'DEM.A.PL',
  'DEF.A.PL',
  'DEF.I.SG',
  'IND.H.SG',
  'IND.I.PL',
  'IND.H.DU',
  'IND.A.PL',
  'DEF.A.DU',
  'IND.I.DU',
  'DEM.H.DU',
  'DEF.H.DU',
  'DEF.I.PL',
  'DEF.H.SG',
  'DEM.A.SG',
  'IND.I.SG',
  'IND.A.DU',
  'DEM.I.PL',
  'DEF.A.SG',
  'DEM.A.DU',
  'IND.H.PL',
  'IND.A.SG'],
 'NOM': ['katana.N2.I.Du',
  'mort.N2.H.Pl',
  u'ch\xe8vre.N1.A.Pl',
  'survivante.N2.H.Pl',
  'peur.N2.I.Sg',
  'autorisation.N1.I.Sg',
  'Rachel.N2.H.Sg',
  'cactus.N2.A.Pl',
  u'fant\xf4me.N1.A.Sg',
  'escargot.N2.A.Pl',
  'cochon.N2.A.Sg',
  'habitant.N1.H.Pl',
  u'fermi\xe8re.N1.H.Sg',
  'grenouille.N1.A.Du',
  'immortelle.N1.H.Sg',
  u'contr\xe9e.N2.I.Sg',
  u'fant\xf4me.N1.A

In [61]:
def sortLevel(level):
    sortedLevel=level
    name="???"
    if "Erg" in level:
        sortedLevel=[f for f in "Erg Abs Dat Obl".split(" ") if f in level]
        name="Cas"
    elif "Acc" in level or "Nom" in level:
        sortedLevel=[f for f in "Nom Acc Dat Obl".split(" ") if f in level]
        name="Cas"
    elif "3Sg" in level:
        sortedLevel=[f for f in u"3Sg 3Du 3Pau 3Pl".split(" ") if f in level]
        name="Pers"
    elif "Sg" in level or "Pl" in level:
        sortedLevel=[f for f in u"Sg Du Pau Pl".split(" ") if f in level]
        name="Nombre"
    elif "Hum" in level or "Ani" in level:
        sortedLevel=[f for f in "Hum Ani Anim Ina Inan".split(" ") if f in level]
        name="Genre"
    elif "H" in level:
        sortedLevel=[f for f in "H A I".split(" ") if f in level]
        name="Genre"
    elif "M" in level or "F" in level or "N" in level:
        sortedLevel=[f for f in "M F N".split(" ") if f in level]
        name="Genre"
    elif "A" in level and "B" in level:
        sortedLevel=[f for f in "A B C D".split(" ") if f in level]
        name="Genre"
    elif "Def" in level:
        sortedLevel=[f for f in "Def Indef Ind Dem".split(" ") if f in level]
        name="DET"

    elif "Prs" in level:
        print level
        sortedLevel=[f for f in "Prs PRS Pst PST Fut FUT".split(" ") if f in level]
        name="Temps"
    elif "V1" in level or "N1" in level or "A1" in level:
        name="CF"
    elif "Vt" in level:
        name="Trans"
    return sortedLevel, name
        
def filledTable(fc,lexeme,rows,cols):
    filled=[f.split(".") for f in fc]
    filled=[f if "hyper" not in f else [c for c in f if c!="hyper"] for f in filled]
    print [f for f in filled if f and len(f)>5]
    for iF,f in enumerate(filled):
        filled[iF]=[c.capitalize() if c[0] not in "123" else c for c in f]
    dfFilled=pd.DataFrame(filled)
    dfFilled.columns=["C"+str(n) for n in dfFilled.columns.tolist()]
    if lexeme!=0:
        dfFilled["lexeme"]=dfFilled["C0"]
        lLexeme="lexeme"
    else:
        lLexeme="C"+str(lexeme)
    lRows=["C"+str(n) for n in rows]
    lCols=["C"+str(n) for n in cols]
#     display(dfFilled)

    
    result=pd.pivot_table(dfFilled,index=lRows,columns=lCols,values=lLexeme,aggfunc="count",dropna = False,fill_value=0)
#     display(result)
    if isinstance(result.index, pd.MultiIndex):
        levels=result.index.levels
        for iL,level in enumerate(levels):
            print iL, level
            sort,name=sortLevel(level)
            result=result.reindex(sort,level=iL)
            result.index=result.index.set_names(name, level=iL)
    else:
        sort,name=sortLevel(result.index)
        result=result.reindex(sort)
        result.index=result.index.set_names(name)

    if isinstance(result.columns, pd.MultiIndex):
        levels=result.columns.levels
        for iL,level in enumerate(levels):
            sort,name=sortLevel(level)
            result=result.reindex(sort,level=iL,axis=1)
            result.columns=result.columns.set_names(name, level=iL)
#             result=result.reindex(sortLevel(level),level=-iL,axis=1)
    else:
        sort,name=sortLevel(result.columns)
        result=result.reindex(sort,axis=1)
        result.columns=result.columns.set_names(name)

#         result=result.reindex(sortLevel(result.columns),axis=1)


    
    return result

In [62]:
tableauxRemplissage={}

# Paramétrage des remplissages

In [63]:
fc=filledCells["NOM"]
print fc[0]
# tableauxRemplissage["NOM"]=filledTable(fc,0,[1],[2])
tableauxRemplissage["NOM"]=filledTable(fc,0,[1],[3])
# tableauxRemplissage["NOM"]=filledTable(fc,0,[2],[3])
# tableauxRemplissage["NOM"]=filledTable(fc,0,[1],[2,3])
# tableauxRemplissage["NOM"]=filledTable(fc,0,[1],[3,4])
# tableauxRemplissage["NOM"]=filledTable(fc,0,[2,1],[3,4])
# tableauxRemplissage["NOM"]=filledTable(fc,0,[2],[3,4])


katana.N2.I.Du
[]


In [64]:
fc=filledCells["VER"]
print fc[0]
tableauxRemplissage["VER"]=filledTable(fc,0,[1],[2,3])
# tableauxRemplissage["VER"]=filledTable(fc,0,[1,2],[3,4])
# tableauxRemplissage["VER"]=filledTable(fc,0,[2,3],[4,5])


manger.Prs.3Du.A
[]
Index([u'Prs', u'Pst'], dtype='object', name=u'C1')


In [65]:
fc=filledCells["ADJ"]
print fc[0]
tableauxRemplissage["ADJ"]=filledTable(fc,0,[1],[2])
# tableauxRemplissage["ADJ"]=filledTable(fc,0,[1],[2,3])
# tableauxRemplissage["ADJ"]=filledTable(fc,0,[1,2],[3,4])

courageux.H.SG
[]


In [66]:
fc=filledCells["DET"]
print fc[0]
tableauxRemplissage["DET"]=filledTable(fc,0,[1],[2])
# tableauxRemplissage["DET"]=filledTable(fc,0,[1],[2,3])
# tableauxRemplissage["DET"]=filledTable(fc,0,[1,2],[3,4])

DEF.I.DU
[]


In [67]:
fc=filledCells["PRO"]
print fc[0]
if fc[0]:
    tableauxRemplissage["PRO"]=filledTable(fc,0,[1],[2])
    # tableauxRemplissage["PRO"]=filledTable(fc,0,[1],[2,3])

PRO.H.Sg
[]


# Tableaux remplis

In [68]:
tableauxLatexRemplissage=[]
tableauxLatexRemplissage.append(ur"\subsection*{Tableaux de remplissage}")
for cat in "DET PRO NOM ADJ VER".split(" "):
#    print ur"\section{%s}"%cat
    if cat in tableauxRemplissage:
        tableauxLatexRemplissage.append(u"%%\\columnbreak\n\\subsubsection*{%s}"%cat)
        tableauxLatexRemplissage.append(tableauxRemplissage[cat].to_latex(multirow=True)+u" \\\\ \\medskip")
        tableauxLatexRemplissage.append("")
print "\n".join(tableauxLatexRemplissage)

\subsection*{Tableaux de remplissage}
%\columnbreak
\subsubsection*{DET}
\begin{tabular}{lrrr}
\toprule
Nombre &  Sg &  Du &  Pl \\
Genre &     &     &     \\
\midrule
H     &   2 &   3 &   3 \\
A     &   3 &   3 &   3 \\
I     &   3 &   2 &   3 \\
\bottomrule
\end{tabular}
 \\ \medskip

%\columnbreak
\subsubsection*{PRO}
\begin{tabular}{lr}
\toprule
Nombre &  Sg \\
Genre &     \\
\midrule
H     &   1 \\
\bottomrule
\end{tabular}
 \\ \medskip

%\columnbreak
\subsubsection*{NOM}
\begin{tabular}{lrrr}
\toprule
Nombre &  Sg &  Du &  Pl \\
CF &     &     &     \\
\midrule
N1 &  16 &   4 &   8 \\
N2 &   9 &   4 &   8 \\
P  &   0 &   0 &   0 \\
\bottomrule
\end{tabular}
 \\ \medskip

%\columnbreak
\subsubsection*{ADJ}
\begin{tabular}{lrrr}
\toprule
Nombre &  Sg &  Du &  Pl \\
Genre &     &     &     \\
\midrule
H     &   1 &   0 &   1 \\
A     &   1 &   1 &   3 \\
I     &   4 &   3 &   1 \\
\bottomrule
\end{tabular}
 \\ \medskip

%\columnbreak
\subsubsection*{VER}
\begin{tabular}{lrrrrrrrrr}

In [69]:
with codecs.open(serie+"Flexions.tex","w",encoding="utf8") as outFile:
    print "\n".join(tableauxLatex+cfLatex+genresLatex+tableauxLatexRemplissage)
    outFile.write("\n".join(tableauxLatex+cfLatex+genresLatex+tableauxLatexRemplissage))

\subsection*{Tableaux de flexion}
\subsubsection*{DET}
\needspace{6\baselineskip}
\noindent
DET DEM => d\\
\begin{tabular}{llll}
\toprule
Nombre &      Sg &      Du &      Pl \\
Genre &         &         &         \\
\midrule
A     &  d-u-to &  d-u-ti &  d-u-te \\
H     &  d-a-to &  d-a-ti &  d-a-te \\
I     &  d-i-to &  d-i-ti &  d-i-te \\
\bottomrule
\end{tabular}
 \\ \medskip

\subsubsection*{PRO}
\needspace{6\baselineskip}
\noindent
PRO PRO => al\\
\begin{tabular}{llll}
\toprule
Nombre &      Sg &      Du &      Pl \\
Genre &         &         &         \\
\midrule
A     &  al-u-s &  al-u-d &  al-u-p \\
H     &  al-o-s &  al-o-d &  al-o-p \\
I     &  al-i-s &  al-i-d &  al-i-p \\
\bottomrule
\end{tabular}
 \\ \medskip

\subsubsection*{NOM}
\needspace{6\baselineskip}
\noindent
N1 caillou => tari\\
\begin{tabular}{llll}
\toprule
Nombre &       Sg &       Du &       Pl \\
\midrule
Forme &  tari-gi &  tari-ma &  tari-lo \\
\bottomrule
\end{tabular}
 \\ \medskip

\needspace{6\baselinesk

In [70]:
ding()